# Test scLIDER Pipeline vs Original Notebook

In [ ]:
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=100, facecolor="white")

## Using Original Notebook Steps (10X Data)

In [ ]:
# Load 10X data - same as original notebook
data_path = "/tmp/rnaseqclassify/droplet/Thymus-10X_P7_11"
adata = sc.read_10x_mtx(data_path, var_names='gene_symbols', cache=True)

print(f"Raw data: {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
# QC metrics
adata.var["mt"] = adata.var_names.str.startswith("Mt-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

print(f"QC calculated")

In [ ]:
# Filter cells
adata = adata[
    (adata.obs['n_genes_by_counts'] > 200) &
    (adata.obs['pct_counts_mt'] < 10)
]

print(f"After QC filter: {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
# Normalize
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

print("Normalized")

In [ ]:
# HVG selection
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

print(f"HVGs: {adata.var['highly_variable'].sum()}")

In [ ]:
# PCA
sc.tl.pca(adata, use_highly_variable=True)

print(f"PCA computed: {adata.obsm['X_pca'].shape}")

In [ ]:
# Neighbors
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca')

print("Neighbors computed")

In [ ]:
# Leiden clustering
sc.tl.leiden(adata, resolution=0.5)

print(f"Leiden clusters: {len(adata.obs['leiden'].unique())}")
print(f"Cluster IDs: {sorted(adata.obs['leiden'].unique())}")

In [ ]:
# UMAP
sc.tl.umap(adata, min_dist=0.1)

print(f"UMAP computed: {adata.obsm['X_umap'].shape}")
print(f"UMAP range X: [{adata.obsm['X_umap'][:, 0].min():.2f}, {adata.obsm['X_umap'][:, 0].max():.2f}]")
print(f"UMAP range Y: [{adata.obsm['X_umap'][:, 1].min():.2f}, {adata.obsm['X_umap'][:, 1].max():.2f}]")

In [ ]:
# Plot UMAP colored by clusters
sc.pl.umap(adata, color='leiden', size=30, legend_loc='on data')

## Now Test Our Python Pipeline

In [ ]:
from scrna_pipeline.pipeline import run_pipeline

adata_pipeline = run_pipeline(
    '/tmp/rnaseqclassify/droplet/Thymus-10X_P7_11',
    n_genes_min=200,
    pct_mt_max=10.0,
    n_top_genes=2000,
    n_pcs=30,
    n_neighbors=15,
    leiden_resolution=0.5,
    umap_min_dist=0.1,
)

print(f"Pipeline result: {adata_pipeline.n_obs} cells x {adata_pipeline.n_vars} genes")
print(f"Clusters: {len(adata_pipeline.obs['leiden'].unique())}")
print(f"UMAP shape: {adata_pipeline.obsm['X_umap'].shape}")

In [ ]:
# Plot pipeline UMAP
sc.pl.umap(adata_pipeline, color='leiden', size=30, legend_loc='on data')

## Compare the Two

In [ ]:
print("Original notebook:")
print(f"  Cells: {adata.n_obs}")
print(f"  Clusters: {len(adata.obs['leiden'].unique())}")
print(f"  UMAP range X: [{adata.obsm['X_umap'][:, 0].min():.2f}, {adata.obsm['X_umap'][:, 0].max():.2f}]")
print(f"  UMAP range Y: [{adata.obsm['X_umap'][:, 1].min():.2f}, {adata.obsm['X_umap'][:, 1].max():.2f}]")

print("\nOur pipeline:")
print(f"  Cells: {adata_pipeline.n_obs}")
print(f"  Clusters: {len(adata_pipeline.obs['leiden'].unique())}")
print(f"  UMAP range X: [{adata_pipeline.obsm['X_umap'][:, 0].min():.2f}, {adata_pipeline.obsm['X_umap'][:, 0].max():.2f}]")
print(f"  UMAP range Y: [{adata_pipeline.obsm['X_umap'][:, 1].min():.2f}, {adata_pipeline.obsm['X_umap'][:, 1].max():.2f}]")